# LFM3 TPU: boundary, chunked-prefill and state-reuse validation

Run in the already-installed Colab TPU environment. These are ordinary Python cells with top-level `await`; no virtual environment, shell Python process, or generated probe script is needed.

1. Restart the Colab **session** before the first run. Keep the existing VM and its files.
2. Leave `MODE = "baseline"` and run the three code cells below. Expect `BASELINE_PASS requests=30`.
3. Restart the session, change only `MODE = "chunked"`, and rerun those cells. Expect `CHUNKED_PASS requests=30`.
4. Share the pass markers or the failed run's log and JSON path. The last cell reads the current log after a crash.

Both runs use one state slot, BF16, a 2048-token context, and disabled compilation/prefix reuse. The baseline uses a 2048-token prefill limit; the second run uses 128-token chunks. Exact input lengths cover page, chunk and 256-token sliding-window boundaries. Each of the 15 prompts runs twice in a different order.

A pass establishes completion and exact generated-token agreement for this workload. It does **not** establish reference-logit accuracy, a correct sliding-window mask in isolation, concurrency, compiled execution, or steady-state performance. A mismatch needs investigation; differing prefill shapes can also change floating-point rounding. Length-capped responses are recorded and compared, not treated as semantic-answer failures.


In [ ]:
from pathlib import Path

# Run baseline first. Restart the Colab session, change to "chunked", and rerun.
MODE = "baseline"
MODEL = "LiquidAI/LFM3-1.7B"
REVISION = "1da06bec6a819da10b54c67e5c90972b46d81871"
REPO = Path("/content/sglang-shortconv")
ARTIFACT_DIR = Path("/content/lfm3-validation")
PROMPT_LENGTHS = [63, 64, 65, 127, 128, 129, 255, 256, 257, 511, 512, 513, 1024]
REPEATS = 2
SAMPLING = {"temperature": 0.0, "max_new_tokens": 64}

assert MODE in ("baseline", "chunked")
ENGINE_ARGS = dict(
    model_path=MODEL,
    revision=REVISION,
    model_config_parser="lfm3",
    model_impl="sglang",
    dtype="bfloat16",
    kv_cache_dtype="bfloat16",
    attention_backend="pallas_tpu_rpa_v3",
    tp_size=1,
    context_length=2048,
    chunked_prefill_size=2048 if MODE == "baseline" else 128,
    max_running_requests=1,
    max_mamba_cache_size=1,
    max_total_tokens=4096,
    disable_radix_cache=True,
    enable_torch_compile=False,
    grammar_backend="none",
    random_seed=1234,
    log_level="info",
)

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import traceback
from datetime import datetime, timezone


def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO), *args], text=True, encoding="utf-8"
    ).strip()


def save_report():
    temporary = report_path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    temporary.replace(report_path)


def record(message):
    line = f"{datetime.now(timezone.utc).isoformat()} {message}"
    log_file.write(line + "\n")
    log_file.flush()
    print(line, flush=True)


def make_cases(tokenizer):
    def chat(text):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": text}],
            tokenize=True,
            add_generation_prompt=True,
            return_dict=False,
            enable_thinking=False,
        )

    question = "List the numbers from 1 to 10, separated by commas."
    cases = {
        "numbers_short": chat(question),
        "capital_short": chat("What is the capital of France?"),
    }
    # Keep the chat framing and the final question intact. Only filler is sliced.
    marker = "__LFM3_FILLER__"
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": marker + "\n\n" + question}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    assert rendered.count(marker) == 1
    before, after = rendered.split(marker)
    prefix = tokenizer.encode(before, add_special_tokens=False)
    suffix = tokenizer.encode(after, add_special_tokens=False)
    filler = tokenizer.encode(
        "This is background text about trees, rivers, and mountains. " * 256,
        add_special_tokens=False,
    )
    for length in PROMPT_LENGTHS:
        count = length - len(prefix) - len(suffix)
        assert 0 <= count <= len(filler), (length, len(prefix), len(suffix))
        ids = prefix + filler[:count] + suffix
        assert len(ids) == length
        assert length + SAMPLING["max_new_tokens"] < ENGINE_ARGS["context_length"]
        cases[f"tokens_{length}"] = ids
    return cases


def signature(result):
    # Timings and request IDs naturally differ; generated content must agree.
    return result["output_ids"], result["text"], result["finish_reason"]


def find_mismatches(current, baseline=None):
    failures = []
    first = {}
    for row in current["results"]:
        name = row["case"]
        if name in first and signature(row) != signature(first[name]):
            failures.append(f"repeat mismatch: {name}, pass {row['pass']}")
        first.setdefault(name, row)
    if baseline is not None:
        expected = {}
        for row in baseline["results"]:
            expected.setdefault(row["case"], row)
        for name, row in first.items():
            if signature(row) != signature(expected[name]):
                failures.append(f"baseline/chunked mismatch: {name}")
    return failures

In [ ]:
# This cell captures file descriptors before importing SGLang or spawning workers.
# Use a fresh Colab session for each mode; files under /content survive a session restart.
if "sglang" in sys.modules:
    raise RuntimeError("Restart the Colab session, then run these cells from the top.")

os.environ["SGLANG_PLATFORM"] = "tpu_oot"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONFAULTHANDLER"] = "1"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
log_path = ARTIFACT_DIR / f"{MODE}-{run_id}.log"
report_path = ARTIFACT_DIR / f"{MODE}-{run_id}.json"
baseline_pointer = ARTIFACT_DIR / "baseline-latest.txt"
baseline = None
if MODE == "chunked":
    baseline = json.loads(Path(baseline_pointer.read_text().strip()).read_text())
    assert baseline["mode"] == "baseline" and baseline["status"] == "passed"

versions = {}
for package in (
    "sglang",
    "sglang-tpu",
    "torch",
    "torch-tpu",
    "jax",
    "jaxlib",
    "transformers",
):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None

report = {
    "schema": 1,
    "mode": MODE,
    "run_id": run_id,
    "status": "running",
    "code_commit": git("rev-parse", "HEAD"),
    "source_diff_sha256": hashlib.sha256(
        git(
            "diff",
            "HEAD",
            "--",
            "python/sglang",
            "sglang_tpu",
            "python/pyproject.toml",
            "pyproject.toml",
        ).encode()
    ).hexdigest(),
    "versions": versions,
    "engine_args": ENGINE_ARGS,
    "sampling": SAMPLING,
    "repeats": REPEATS,
    "baseline_report": str(baseline_pointer.read_text().strip()) if baseline else None,
    "log_path": str(log_path),
    "cases": {},
    "results": [],
}
if baseline:
    for key in (
        "schema",
        "code_commit",
        "source_diff_sha256",
        "versions",
        "sampling",
        "repeats",
    ):
        assert report[key] == baseline[key], (
            f"Baseline differs in {key}; rerun baseline."
        )
    comparable = lambda args: {
        k: v for k, v in args.items() if k != "chunked_prefill_size"
    }
    assert comparable(ENGINE_ARGS) == comparable(baseline["engine_args"])
save_report()
(ARTIFACT_DIR / "last-report.txt").write_text(str(report_path), encoding="utf-8")
Path("/content/lfm3-last-run.txt").write_text(str(log_path), encoding="utf-8")
print("Mode:", MODE, "\nLog:", log_path, "\nReport:", report_path)

log_file = log_path.open("a", encoding="utf-8", buffering=1)
saved_stdout, saved_stderr = os.dup(1), os.dup(2)
engine = None
try:
    os.dup2(log_file.fileno(), 1)
    os.dup2(log_file.fileno(), 2)
    record(f"RUN_BEGIN mode={MODE} commit={report['code_commit']}")
    record(f"VERSIONS {versions}")
    record(f"ENGINE_ARGS {ENGINE_ARGS}")

    import sglang
    import sglang_tpu

    for module in (sglang, sglang_tpu):
        location = Path(module.__file__).resolve()
        record(f"IMPORT {module.__name__}: {location}")
        assert location.is_relative_to(REPO.resolve()), f"Wrong checkout: {location}"

    engine = sglang.Engine(**ENGINE_ARGS)
    record("ENGINE_OK")
    tokenizer = engine.tokenizer_manager.tokenizer
    assert tokenizer is not None
    cases = make_cases(tokenizer)
    report["cases"] = cases
    if baseline:
        assert cases == baseline["cases"], "Input tokens differ; rerun baseline."
    save_report()
    names = list(cases)
    record(f"CASES {[(name, len(ids)) for name, ids in cases.items()]}")

    for repeat in range(REPEATS):
        # Mix short and long requests between repeated uses of the sole state slot.
        order = names if repeat == 0 else names[::2] + names[1::2]
        for name in order:
            ids = cases[name]
            record(
                f"REQUEST_BEGIN pass={repeat + 1} case={name} prompt_tokens={len(ids)}"
            )
            result = await engine.async_generate(
                input_ids=ids, sampling_params=SAMPLING
            )
            meta = result["meta_info"]
            row = {
                "pass": repeat + 1,
                "case": name,
                "text": result["text"],
                "output_ids": result["output_ids"],
                "finish_reason": meta["finish_reason"],
                "meta_info": meta,
            }
            report["results"].append(row)
            save_report()
            record(
                f"REQUEST_END pass={repeat + 1} case={name} "
                f"finish={row['finish_reason']} completion_tokens={meta['completion_tokens']} "
                f"cached_tokens={meta['cached_tokens']} text={row['text']!r}"
            )
            assert meta["prompt_tokens"] == len(ids), (name, meta)
            assert meta["cached_tokens"] == 0, (name, meta)
            assert meta["finish_reason"]["type"] in ("stop", "length"), (name, meta)
            assert meta["completion_tokens"] > 0 and row["output_ids"], (name, meta)

    failures = find_mismatches(report, baseline)
    report["mismatches"] = failures
    assert len(report["results"]) == REPEATS * len(cases)
    report["status"] = "failed" if failures else "passed"
    save_report()
    for failure in failures:
        record("MISMATCH " + failure)
    if failures:
        raise AssertionError(
            f"{len(failures)} output mismatches; inspect the JSON report."
        )
    if MODE == "baseline":
        baseline_pointer.write_text(str(report_path), encoding="utf-8")
    record(f"{MODE.upper()}_PASS requests={len(report['results'])}")
except BaseException:
    report["status"] = "failed"
    report["error"] = traceback.format_exc()
    save_report()
    record(report["error"])
    raise
finally:
    try:
        if engine is not None:
            record("ENGINE_SHUTDOWN_BEGIN")
            engine.shutdown()
            record("ENGINE_SHUTDOWN_END")
    finally:
        os.dup2(saved_stdout, 1)
        os.dup2(saved_stderr, 2)
        os.close(saved_stdout)
        os.close(saved_stderr)
        log_file.close()

## Read the latest log
The pointer is written before engine startup. Reports are saved after every response, and each run gets a unique filename.


In [ ]:
# Run this after reconnecting if a worker failure interrupted the notebook.
from pathlib import Path

path = Path(Path("/content/lfm3-last-run.txt").read_text().strip())
print("Log:", path)
print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-250:]))